In [ ]:
!pip install crewai dotenv langchain-google-genai crewai-tools

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os 
from dotenv import load_dotenv
import crewai 
from crewai import Agent,Task,Crew
from langchain_google_genai import ChatGoogleGenerativeAI
from crewai_tools import SerperDevTool
load_dotenv(r"C:\Users\medhu\Desktop\Gemini\.env.txt")
api_key=os.getenv("GOOGLE_API_KEY")
print("GOOGLE_API_KEY is loaded:",api_key is not None)
serper_api_key=os.getenv("SERPER_API_KEY")
print("SERPER_API_KEY loaded:",serper_api_key is not None)

In [ ]:
llm=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key
)
search_tool=SerperDevTool(
    api_key=serper_api_key
)

In [ ]:
check = Agent(
    role="Research Agent",
    goal="To search the latest mobiles under 20k with 5G network",
    backstory="You are an Mobile researcher specialist where you know RAM,ROM,Processor,Battery power,OS in mobiles",
    llm=llm,
    tools=[search_tool],
    verbose=True
)

In [ ]:
compare=Agent(
    role="Comparison Agent",
    goal="To compare with other mobiles and selecting the best mobiles",
    backstory="You are an Mobile researcher, You knows the specifications of all mobiles, You are specialist where you know RAM,ROM,Processor,Battery power,OS in mobiles",
    llm=llm,
    verbose=True
)

In [ ]:
final=Agent(
    role="Suggestion Agent",
    goal="Suggest one mobile that is best among all mobiles under 20k",
    backstory="You are an Mobile researcher, You knows the specifications of all mobiles, You are specialist where you know RAM,ROM,Processor,Battery power,OS in mobiles",
    llm=llm,
    verbose=True
)

In [ ]:
check_task=Task(
    description="""
    Research mobiles under 20k with its
    -Mobile brand name
    -display size
    -RAM,ROM
    -camera quality
    -specialities
    -OS
    """,
    expected_output="""
    Select 10 latest mobiles under 20k 
    with all its specifications""",
    agent=check
)

In [ ]:
compare_task=Task(
    description="""
    Among 10 mobiles compare the best mobile with all the features 
    -Best camera quality
    -Best Display
    -Effective RAM,ROM
    -Best cost
    -Best battery
    -Best Processor
    """,
    expected_output="""
    Compare and give 3 best mobiles with all specifications
    -Mobile brand name
    -display size
    -RAM,ROM
    -camera quality
    -specialities
    -OS
    """,
    agent=compare,
    context=[check_task]
)

In [ ]:
final_task=Task(
    description="""
    Select one best mobile among 3 mobiles with best specifications under 20k on the basis of 
    -Best camera quality
    -Best Display
    -Effective RAM,ROM
    -Best cost
    -Best battery
    -Best Processor
    """,
    expected_output="""
    Give the latest mobile under 20k to buy with all details- Brand Name
    - Model Name
    - Price
    - Display Size
    - RAM
    - Storage
    - Processor
    - Camera Specifications
    - Battery Capacity
    - Operating System
    - Special Features
    - Overall Ranking
    """,
    agent=final,
    context=[compare_task]
)

In [ ]:
crew=Crew(
    agents=[check,compare,final],
    tasks=[check_task,compare_task,final_task],
    verbose=2
)
result=crew.kickoff()
print(result)